# GP-v2 corpus — live composition & balance

**Re-runnable**: reads the cut manifests on disk (axes still being produced show as *pending*),
so *Run All* refreshes the picture as the campaign progresses. Method + rationale:
[`BG_ASR_DATA_PIPELINE.md`](BG_ASR_DATA_PIPELINE.md) (preprocessing per axis, quality gates,
the 0.5–2× register-balance rule and the mux-weights lever).

## 1 · Scan the axis manifests

In [1]:
from pathlib import Path
from lhotse import CutSet

REPO = Path.home() / "WORK/OmniVoice"

def hours_split(path, splitter=None):
    # total hours (and optional sub-splits by cut-id prefix) of a cuts manifest
    if not path.exists():
        return None
    tot, sub = 0.0, {}
    for c in CutSet.from_file(path):
        d = c.duration / 3600
        tot += d
        if splitter:
            sub[splitter(c.id)] = sub.get(splitter(c.id), 0.0) + d
    return tot, sub

# v1 gold mixes two registers -> split by cut-id prefix
v1 = hours_split(REPO/"data/fbank/bg_cuts_train.jsonl.gz",
                 lambda cid: "parliament" if cid.startswith("plsten") else "audiobook")

AXES = [  # (label, register, manifest, hours)
    ("v1 parliament (gold)",   "parliament",     "bg_cuts_train.jsonl.gz [plsten]",
        None if v1 is None else v1[1].get("parliament", 0.0)),
    ("v1 AB400 audiobook (gold)", "read literary", "bg_cuts_train.jsonl.gz [rest]",
        None if v1 is None else v1[1].get("audiobook", 0.0)),
    ("parliament recovery (gold)", "parliament",  "fbank_reseg/bg_cuts_train.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank_reseg/bg_cuts_train.jsonl.gz"))),
    ("BNR archive (weak)",     "archival radio", "bg_cuts_bnr.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_bnr.jsonl.gz"))),
    ("DnevenRed (weak+agree)", "conversational", "bg_cuts_ytdr.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_ytdr.jsonl.gz"))),
    ("Bloomberg BG (weak+agree)", "broadcast news", "bg_cuts_ytbb.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_ytbb.jsonl.gz"))),
    ("Govori Internet (weak+agree)", "conversational", "bg_cuts_ytgi.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_ytgi.jsonl.gz"))),
    ("FLEURS train (gold-read)", "read modern",   "bg_cuts_fleurs_train.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_fleurs_train.jsonl.gz"))),
    ("CommonVoice train (gold-read)", "read modern", "bg_cuts_cv_train.jsonl.gz",
        (lambda r: r and r[0])(hours_split(REPO/"data/fbank/bg_cuts_cv_train.jsonl.gz"))),
]

present = [(l, reg, h) for l, reg, _, h in AXES if h]
pending = [(l, reg) for l, reg, _, h in AXES if not h]
tot = sum(h for _, _, h in present)
print(f"{'axis':34s} {'register':16s} {'hours':>8s} {'share':>7s}")
for l, reg, h in present:
    print(f"{l:34s} {reg:16s} {h:8.1f} {100*h/tot:6.1f}%")
print(f"{'TOTAL (present axes)':34s} {'':16s} {tot:8.1f}")
for l, reg in pending:
    print(f"{l:34s} {reg:16s} {'pending':>8s}")

axis                               register            hours   share
v1 parliament (gold)               parliament          238.9   21.7%
v1 AB400 audiobook (gold)          read literary       410.3   37.2%
parliament recovery (gold)         parliament          334.4   30.3%
BNR archive (weak)                 archival radio      102.9    9.3%
FLEURS train (gold-read)           read modern           9.5    0.9%
CommonVoice train (gold-read)      read modern           7.0    0.6%
TOTAL (present axes)                                  1103.0
DnevenRed (weak+agree)             conversational    pending
Bloomberg BG (weak+agree)          broadcast news    pending
Govori Internet (weak+agree)       conversational    pending


## 2 · Composition chart

In [2]:
import matplotlib.pyplot as plt

labels = [l for l, _, _ in present][::-1]
vals   = [h for _, _, h in present][::-1]
fig, ax = plt.subplots(figsize=(9, 0.5*len(labels)+1.2))
bars = ax.barh(labels, vals, color="#2b6cb0", height=0.55)
for b, v in zip(bars, vals):                       # selective direct labels, text in ink not series color
    ax.text(b.get_width()+4, b.get_y()+b.get_height()/2, f"{v:.0f} h",
            va="center", fontsize=9, color="#333333")
ax.set_xlabel("hours (post-filter)")
ax.set_title(f"GP-v2 training corpus — {tot:.0f} h across {len(present)} axes"
             + (f"  ({len(pending)} pending)" if pending else ""))
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)                      # recessive grid
ax.margins(x=0.12)
fig.tight_layout()
fig.savefig(Path.home()/"WORK/OmniVoice/docs/bg_asr_corpus.png", dpi=110)
plt.show()

/tmp/ipykernel_1869951/1631200143.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3 · Register balance vs the 0.5–2× band

In [3]:
reg_h = {}
for _, reg, h in present:
    reg_h[reg] = reg_h.get(reg, 0.0) + h
mean = sum(reg_h.values())/len(reg_h)
print(f"{'register':16s} {'hours':>8s} {'share':>7s}   vs-mean")
for reg, h in sorted(reg_h.items(), key=lambda x: -x[1]):
    r = h/mean
    flag = "OK" if 0.5 <= r <= 2.0 else "OUT-OF-BAND"
    print(f"{reg:16s} {h:8.1f} {100*h/tot:6.1f}%   {r:4.1f}x  {flag}")
mx, mn = max(reg_h.values()), min(reg_h.values())
print(f"\nmax/min register spread: {mx/mn:.1f}x  (guideline: keep registers within ~0.5-2x of "
      f"each other; GP-v2 keeps the NATURAL mix for data-effect attribution — rebalance, if "
      f"eval demands it, via train-time mux weights, not re-merging)")

register            hours   share   vs-mean
parliament          573.3   52.0%    2.1x  OUT-OF-BAND
read literary       410.3   37.2%    1.5x  OK
archival radio      102.9    9.3%    0.4x  OUT-OF-BAND
read modern          16.5    1.5%    0.1x  OUT-OF-BAND

max/min register spread: 34.7x  (guideline: keep registers within ~0.5-2x of each other; GP-v2 keeps the NATURAL mix for data-effect attribution — rebalance, if eval demands it, via train-time mux weights, not re-merging)


## 4 · The rebalancing lever (GP-v3, if needed)

In [4]:
# Example: weights that would sample registers toward uniform shares WITHOUT duplicating data.
# (Illustrative — wire into the datamodule only if GP-v2's eval shows an under-served register.)
target = {reg: 1/len(reg_h) for reg in reg_h}           # uniform target shares
w = {reg: target[reg]/(h/tot) for reg, h in reg_h.items()}
print("register            natural-share  ->  mux weight (relative)")
for reg in sorted(w, key=lambda r: -reg_h[r]):
    print(f"  {reg:16s} {100*reg_h[reg]/tot:6.1f}%          {w[reg]:.2f}")
print("\nusage: CutSet.mux(*axis_cutsets, weights=[...], stop_early=False)  # lazy, no disk cost")

register            natural-share  ->  mux weight (relative)
  parliament         52.0%          0.48
  read literary      37.2%          0.67
  archival radio      9.3%          2.68
  read modern         1.5%          16.69

usage: CutSet.mux(*axis_cutsets, weights=[...], stop_early=False)  # lazy, no disk cost
